In [1]:
import os
import ast
import jax
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc

from tqdm.auto import tqdm

import cfp.preprocessing as cfpp
from cfp.metrics import compute_metrics, compute_mean_metrics, compute_metrics_fast

/home/icb/dominik.klein/mambaforge/envs/cfp/lib/python3.11/site-packages/optuna/study/_optimize.py:29: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from optuna import progress_bar as pbar_module


In [2]:
def get_mask(x, y):
    return x[:, [gene in y for gene in adata_train.var_names]]

In [3]:
def extract_metainfo(file_path):
    """
    Extracts the configuration dictionary, results file path, split index, 
    and wandb run name from the log file and returns them in a dictionary.

    Args:
        file_path (str): Path to the log file.

    Returns:
        dict: A dictionary containing the extracted information:
            - 'config': The configuration dictionary parsed from the log file.
            - 'results_path': The path of the results file.
            - 'split_index': The split index as an integer.
            - 'wandb_run_name': The wandb run name as a string.
    """
    with open(file_path, "r") as file:
        lines = file.readlines()

    # Initialize the metainfo dictionary
    metainfo = {
        "config": None,
        "path_predictions": None,
        "split_index": None,
        "wandb_run_name": None
    }

    # Extract the third line containing the config dictionary
    config_line = lines[2].strip()

    # Parse the configuration dictionary
    try:
        metainfo["config"] = ast.literal_eval(config_line)
    except (SyntaxError, ValueError) as e:
        raise ValueError("Failed to parse configuration dictionary.") from e

    # Extract the results file path and wandb run name
    for line in lines:
        if "Saving results at:" in line:
            metainfo["path_predictions"] = line.split("Saving results at:")[-1].strip()
        # if "🚀 View run" in line:
        #     metainfo["wandb_run_name"] = line.split("View run")[1].split("at:")[0].strip()
        if "🚀 View run" in line:
            # Extract the run name and remove any icons or extra spaces
            raw_run_name = line.split("View run")[1].split("at:")[0].strip()
            metainfo["wandb_run_name"] = raw_run_name.replace("\x1b[33m", "").replace("\x1b[0m", "").strip()

    if not metainfo["path_predictions"]:
        raise ValueError("Results path not found in the log file.")
    if not metainfo["wandb_run_name"]:
        raise ValueError("wandb run name not found in the log file.")

    # Extract the split index from the first line
    for line in lines:
        if line.startswith("split:"):
            try:
                metainfo["split_index"] = int(line.split(":")[-1].strip())
            except ValueError:
                raise ValueError("Failed to parse the split index.")
            break

    if metainfo["split_index"] is None:
        raise ValueError("Split index not found in the log file.")

    return metainfo

# # Example usage
# log_file_path = "path_to_your_log_file.txt"
# metainfo = extract_metainfo(log_file_path)

# # Print the results
# print("Metainfo:", metainfo)


In [4]:
path_log_file = "/home/haicu/soeren.becker/repos/ot_pert_reproducibility/runs_otfm/bash_scripts/h-otfm-norman_29797689.out"

In [5]:
metainfo = extract_metainfo(path_log_file)
config = metainfo["config"]
path_predictions = metainfo["path_predictions"]
split = metainfo["split_index"]
wandb_run_name = metainfo["wandb_run_name"]
print("wandb_run_name", wandb_run_name)
print("split", split)
print("path_predictions", path_predictions)
print("config", config)
assert split == config["dataset"]["split"]


wandb_run_name different-spaceship-245
split 4
path_predictions /lustre/groups/ml01/workspace/ot_perturbation/data/norman_soren/cellflow/out/different-spaceship-245_adata_test_with_predictions_4.h5ad
config {'dataset': {'split': 4, 'sample_rep': 'X_pca', 'perturbation_covariates': {'target_gene': ['gene_1', 'gene_2']}, 'perturbation_covariate_reps': {'target_gene': 'esm2'}, 'wandb_project': 'otfm_norman'}, 'model': {'condition_embedding_dim': 1024, 'time_encoder_dims': [2048, 2048, 2048], 'time_encoder_dropout': 0.0, 'hidden_dims': [4096, 4096, 4096], 'hidden_dropout': 0.0, 'decoder_dims': [4096, 4096, 4096], 'decoder_dropout': 0.2, 'pooling': 'attention_token', 'layers_before_pool': {'target_gene': {'layer_type': 'mlp', 'dims': [1024, 1024], 'dropout_rate': 0.5}}, 'layers_after_pool': {'layer_type': 'mlp', 'dims': [1024, 1024], 'dropout_rate': 0.2}, 'cond_output_dropout': 0.9, 'time_freqs': 1024, 'flow_noise': 1.0, 'learning_rate': 5e-05, 'multi_steps': 20, 'epsilon': 0.1, 'tau_a': 1.

In [6]:
wandb_run_name

'different-spaceship-245'

In [7]:
DATA_DIR = "/home/haicu/soeren.becker/repos/ot_pert_reproducibility/norman2019/norman_preprocessed_adata"

adata_train_path = os.path.join(DATA_DIR, f"adata_train_pca_50_split_{split}.h5ad")
adata_test_path = os.path.join(DATA_DIR, f"adata_val_pca_50_split_{split}.h5ad")
adata_ood_path = os.path.join(DATA_DIR, f"adata_test_pca_50_split_{split}.h5ad")

# load data splits
adata_train = sc.read(adata_train_path)
adata_test = sc.read(adata_test_path)
adata_ood = sc.read(adata_ood_path)

In [8]:
# adata_pred_ood = sc.read_h5ad(f"/lustre/groups/ml01/workspace/ot_perturbation/data/norman_soren/cellflow/out/solar-pine-515_adata_test_with_predictions_0.h5ad")
# path_predictions = f"/lustre/groups/ml01/workspace/ot_perturbation/data/norman_soren/cellflow/out/astral-water-224_adata_test_with_predictions_0.h5ad"
adata_pred_ood = sc.read_h5ad(path_predictions)
adata_pred_ood.obs.loc[:, ["gene_1", "gene_2"]] = adata_pred_ood.obs.condition.str.split("+", expand=True).rename({0: "gene_1", 1: "gene_2"}, axis=1).values
adata_pred_ood.X = adata_pred_ood.layers['X_recon_pred']

In [9]:
adata_pred_ood.X.max(),  adata_ood.X.max(), adata_train.X.max(), adata_test.X.max()

(8.895711, 8.90458, 8.820025, 8.853678)

In [10]:
def add_subgroup_annotations(adata_train, adata): 

    train_conditions = adata_train.obs.condition.str.replace("+ctrl", "").str.replace("ctrl+", "").unique()

    assert not adata[adata.obs.condition != "ctrl"].obs.condition.isin(train_conditions).any()

    mask_single_perturbation = adata.obs.condition.str.contains("ctrl")
    mask_double_perturbation_seen_0 = (
        ~adata.obs.condition.str.contains("ctrl") & 
        ~adata.obs.gene_1.isin(train_conditions) & 
        ~adata.obs.gene_2.isin(train_conditions)
    )
    mask_double_perturbation_seen_1 = (
        ~adata.obs.condition.str.contains("ctrl") & 
        (
            (adata.obs.gene_1.isin(train_conditions) & ~adata.obs.gene_2.isin(train_conditions)) | 
            (~adata.obs.gene_1.isin(train_conditions) & adata.obs.gene_2.isin(train_conditions))
        )
    )
    mask_double_perturbation_seen_2 = (
        ~adata.obs.condition.str.contains("ctrl") & 
        adata.obs.gene_1.isin(train_conditions) & 
        adata.obs.gene_2.isin(train_conditions)
    )
    adata.obs.loc[mask_single_perturbation, "subgroup"] = "single"
    adata.obs.loc[mask_double_perturbation_seen_0, "subgroup"] = "double_seen_0"
    adata.obs.loc[mask_double_perturbation_seen_1, "subgroup"] = "double_seen_1"
    adata.obs.loc[mask_double_perturbation_seen_2, "subgroup"] = "double_seen_2"

add_subgroup_annotations(adata_train, adata_ood)
add_subgroup_annotations(adata_train, adata_pred_ood)

display(adata_ood.obs.subgroup.value_counts())
display(adata_pred_ood.obs.subgroup.value_counts())

subgroup
double_seen_1    14554
single           13520
double_seen_2     3934
double_seen_0     1587
Name: count, dtype: int64

subgroup
double_seen_1    27500
single           20500
double_seen_2     6500
double_seen_0     4500
Name: count, dtype: int64

In [12]:
ood_data_target_decoded = {}
ood_data_target_decoded_predicted = {}

subgroups = ["single", "double_seen_0", "double_seen_1", "double_seen_2"]

for subgroup in tqdm(subgroups):

    ood_data_target_decoded_predicted[subgroup] = {}
    ood_data_target_decoded[subgroup] = {}
    
    for cond in adata_ood.obs["condition"].cat.categories:
        if cond == "ctrl":
            continue
        
        select = adata_ood.obs["condition"] == cond
        select_pred = adata_pred_ood.obs["condition"] == cond

        if subgroup != "all":
            select = select & (adata_ood.obs.subgroup == subgroup)
            select_pred = select_pred & (adata_pred_ood.obs.subgroup == subgroup)

        if not any(select):
            # the condition is not part of this subgroup
            continue
        
        # gene space
        
        ood_data_target_decoded[subgroup][cond] = np.asarray(adata_ood[select].X.todense())
        ood_data_target_decoded_predicted[subgroup][cond] = adata_pred_ood[select_pred].X

100%|██████████| 4/4 [00:02<00:00,  1.34it/s]


In [13]:
def compute_l2(x,y):
    x_mean = x.mean(axis=0)
    y_mean = y.mean(axis=0)
    return np.sqrt(np.sum((x_mean - y_mean)**2))

ood_metrics_decoded = {}

for subgroup in tqdm(subgroups[::-1]):

    print(f"subgroup: {subgroup}")
    print("Computing ood_metrics_decoded")
    # ood set: evaluation in decoded (=gene) space
    ood_metrics_decoded[subgroup] = jax.tree_util.tree_map(
        compute_l2, 
        ood_data_target_decoded[subgroup], 
        ood_data_target_decoded_predicted[subgroup]
    )
    

 25%|██▌       | 1/4 [00:00<00:00,  7.23it/s]

subgroup: double_seen_2
Computing ood_metrics_decoded
subgroup: double_seen_1
Computing ood_metrics_decoded


100%|██████████| 4/4 [00:00<00:00, 13.46it/s]

subgroup: double_seen_0
Computing ood_metrics_decoded
subgroup: single
Computing ood_metrics_decoded


In [14]:
ood_metrics_decoded

{'double_seen_2': {'CDKN1B+CDKN1A': 2.0626736,
  'CDKN1C+CDKN1A': 2.186634,
  'CEBPE+SPI1': 7.222386,
  'DUSP9+IGDCC3': 2.1640472,
  'DUSP9+MAPK1': 2.0845234,
  'FOSB+OSR2': 4.1261683,
  'FOXA1+FOXF1': 2.02112,
  'IRF1+SET': 5.0306616,
  'MAPK1+PRTG': 1.7250568,
  'POU3F2+FOXL2': 2.7135792,
  'UBASH3B+OSR2': 1.8988954,
  'ZC3HAV1+HOXC13': 2.6687205,
  'ZNF318+FOXL2': 1.9313962},
 'double_seen_1': {'AHR+KLF1': 5.104707,
  'BPGM+ZBTB1': 2.649614,
  'C3orf72+FOXL2': 3.507424,
  'CEBPB+MAPK1': 2.3792746,
  'CEBPB+OSR2': 4.278335,
  'CEBPB+PTPN12': 3.0691257,
  'CEBPE+CEBPA': 6.755897,
  'CEBPE+CEBPB': 3.9158468,
  'CEBPE+KLF1': 6.3652706,
  'CEBPE+RUNX1T1': 3.013736,
  'CNN1+MAPK1': 5.141919,
  'DUSP9+KLF1': 5.9956117,
  'ETS2+IKZF3': 6.378536,
  'ETS2+MAP7D1': 2.7778027,
  'FEV+MAP7D1': 3.0935543,
  'FOSB+CEBPB': 3.7213643,
  'FOSB+IKZF3': 4.5110793,
  'FOXA1+HOXB9': 2.6022344,
  'FOXF1+HOXB9': 2.2184749,
  'FOXL2+HOXB9': 3.3709266,
  'IGDCC3+ZBTB25': 3.3391786,
  'JUN+CEBPA': 10.888963,


In [18]:
dfs = []
for subgroup, metrics in ood_metrics_decoded.items():
    df_tmp = pd.DataFrame.from_dict(metrics, orient="index")
    df_tmp["subgroup"] = subgroup
    dfs.append(df_tmp)

df = pd.concat(dfs)
df.head()

In [19]:
df = pd.concat(dfs)
df.head()

,0,subgroup
CDKN1B+CDKN1A,2.062674,double_seen_2
CDKN1C+CDKN1A,2.186634,double_seen_2
CEBPE+SPI1,7.222386,double_seen_2
DUSP9+IGDCC3,2.164047,double_seen_2
DUSP9+MAPK1,2.084523,double_seen_2


In [20]:
out_dir = "/lustre/groups/ml01/workspace/ot_perturbation/data/norman_2/cellflow_l2"

In [21]:
df.to_csv(os.path.join(out_dir, f"l2_split_{split}.csv"))